In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
print("API Key Loaded:", api_key is not None)

API Key Loaded: True


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)
response = llm.invoke("Explain Machine Learning in one sentence")
print(response.content)

c:\Users\kotip\Desktop\GenAI_Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Machine learning enables computers to learn from data, identify patterns, and make decisions or predictions without being explicitly programmed for each task.


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("C:\\Users\\kotip\\Desktop\\GenAI_Chatbot\\data\\sample.pdf")
documents = loader.load()
print(f"Number of pages: {len(documents)}")

C:\Users\kotip\AppData\Local\Temp\ipykernel_17500\3388418017.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 28


In [4]:
documents[0].page_content[:1000]

'International Journal of Social Robotics (2023) 15:473–500\nhttps://doi.org/10.1007/s12369-021-00785-7\nCommunication Models in Human–Robot Interaction: An Asymmetric\nMODel of ALterity in Human–Robot Interaction (AMODAL-HRI)\nHelena Anna Frijns 1 · Oliver Schürer 2 · Sabine Theresia Koeszegi 1\nAccepted: 10 April 2021 / Published online: 3 May 2021\n© The Author(s) 2021\nAbstract\nWe argue for an interdisciplinary approach that connects existing models and theories in Human–Robot Interaction (HRI) to\ntraditions in communication theory. In this article, we review existing models of interpersonal communication and interaction\nmodels that have been applied and developed in the contexts of HRI and social robotics. We argue that often, symmetric models\nare proposed in which the human and robot agents are depicted as having similar ways of functioning (similar capabilities,\ncomponents, processes). However, we argue that models of human–robot interaction or communication should be asymm

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 203


In [6]:
print(chunks[0].page_content)

International Journal of Social Robotics (2023) 15:473–500
https://doi.org/10.1007/s12369-021-00785-7
Communication Models in Human–Robot Interaction: An Asymmetric
MODel of ALterity in Human–Robot Interaction (AMODAL-HRI)
Helena Anna Frijns 1 · Oliver Schürer 2 · Sabine Theresia Koeszegi 1
Accepted: 10 April 2021 / Published online: 3 May 2021
© The Author(s) 2021
Abstract
We argue for an interdisciplinary approach that connects existing models and theories in Human–Robot Interaction (HRI) to
traditions in communication theory. In this article, we review existing models of interpersonal communication and interaction
models that have been applied and developed in the contexts of HRI and social robotics. We argue that often, symmetric models
are proposed in which the human and robot agents are depicted as having similar ways of functioning (similar capabilities,
components, processes). However, we argue that models of human–robot interaction or communication should be asymmetric


In [7]:
print("Chunk 1:")
print(chunks[0].page_content[:300])
print("\n" + "="*50 + "\n")
print("Chunk 2:")
print(chunks[1].page_content[:300])

Chunk 1:
International Journal of Social Robotics (2023) 15:473–500
https://doi.org/10.1007/s12369-021-00785-7
Communication Models in Human–Robot Interaction: An Asymmetric
MODel of ALterity in Human–Robot Interaction (AMODAL-HRI)
Helena Anna Frijns 1 · Oliver Schürer 2 · Sabine Theresia Koeszegi 1
Accepted


Chunk 2:
components, processes). However, we argue that models of human–robot interaction or communication should be asymmetric
instead. We propose an asymmetric interaction model called AMODAL-HRI (an Asymmetric MODel of ALterity in Human–
Robot Interaction). This model is based on theory on joint action, c


In [8]:
# Google's free tier is limiting how many embedding requests can be made per minute.
'''
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
'''

'\nfrom langchain_google_genai import GoogleGenerativeAIEmbeddings\n\nembeddings = GoogleGenerativeAIEmbeddings(\n    model="models/gemini-embedding-001"\n)\n'

In [9]:
# So I am using a local embedding temporarly

from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\kotip\AppData\Local\Temp\ipykernel_17500\3688705741.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1823.13it/s]


In [10]:
sample_embedding = embeddings.embed_query(
    "What is Machine Learning?"
)
print(len(sample_embedding))

384


In [11]:
print(sample_embedding[:10])

[-0.019954534247517586, 0.009878044947981834, 0.010249584913253784, 0.029553698375821114, 0.02718646638095379, -0.019296543672680855, -0.024129576981067657, -0.03773573786020279, -0.041054219007492065, -0.0014749967958778143]


In [12]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="vector_db"
)
print("Vector database created successfully!")

Vector database created successfully!


In [13]:
print(vector_store._collection.count())

812


In [23]:
retriever = vector_store.as_retriever(search_kwargs={"k": 8})   #Tried with 3 chunks- didn't give accurate answer.
results = retriever.invoke("What is OASIS?")
print(results[0].page_content)

Seibt, using intentionalist vocabulary to describe robots is
confusing, inaccurate and imprecise. She proposes OASIS,
the Ontology of Asymmetric Social Interactions, which
provides a description language for further theory devel-
opments, and offers the possibility to include non-humans
as social interaction partners while emphasizing differences
with human social interaction. Seibt argues that in order
to “work with” robots, for instance, robots would need to
be capable of having the phenomenological experience of
working [ 77]. Instead of arguing that a robot is capable of
communication, we can apply Seibt’s framework and say
that the robot is either functionally replicating, imitating,
mimicking, displaying or approximating the process of com-
municating. In the context of this article, however, we are
mostly concerned with the human–robot system and the way
123


In [24]:
'''
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)
'''

'\nfrom langchain.prompts import PromptTemplate\nfrom langchain.chains import RetrievalQA\n\nqa_chain = RetrievalQA.from_chain_type(\n    llm=llm,\n    retriever=retriever,\n    chain_type="stuff"\n)\n'

In [25]:
results = retriever.invoke("What is OASIS?")
context = "\n\n".join([doc.page_content for doc in results])
print(context[:1000])

Seibt, using intentionalist vocabulary to describe robots is
confusing, inaccurate and imprecise. She proposes OASIS,
the Ontology of Asymmetric Social Interactions, which
provides a description language for further theory devel-
opments, and offers the possibility to include non-humans
as social interaction partners while emphasizing differences
with human social interaction. Seibt argues that in order
to “work with” robots, for instance, robots would need to
be capable of having the phenomenological experience of
working [ 77]. Instead of arguing that a robot is capable of
communication, we can apply Seibt’s framework and say
that the robot is either functionally replicating, imitating,
mimicking, displaying or approximating the process of com-
municating. In the context of this article, however, we are
mostly concerned with the human–robot system and the way
123

Seibt, using intentionalist vocabulary to describe robots is
confusing, inaccurate and imprecise. She proposes OASIS,
the

In [26]:
prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
What is OASIS?
"""

In [27]:
response = llm.invoke(prompt)
print(response.content)

OASIS is the Ontology of Asymmetric Social Interactions. It provides a description language for further theory developments and offers the possibility to include non-humans as social interaction partners while emphasizing differences with human social interaction.


In [29]:
#Testing the response
question = "What is OASIS?"
results = retriever.invoke(question)
context = "\n\n".join([doc.page_content for doc in results])
prompt = f"""
Answer using only the context.
Context:
{context}
Question:
{question}
"""
response = llm.invoke(prompt)
print(response.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 29.209581508s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [22]:
questions = [
    "What is OASIS?",
    "What is human robot interaction?",
    "What does the paper say about communication?",
    "What are social interactions?",
    "Summarize the paper in 5 points"
]
for q in questions:
    print("="*50)
    print("QUESTION:", q)
    results = retriever.invoke(q)
    context = "\n\n".join([doc.page_content for doc in results])
    prompt = f"""
    Answer using only the context.
    Context:
    {context}
    Question:
    {q}
    """
    response = llm.invoke(prompt)
    print(response.content)

QUESTION: What is OASIS?
OASIS is the Ontology of Asymmetric Social Interactions. It provides a description language for further theory developments and offers the possibility to include non-humans as social interaction partners while emphasizing differences with human social interaction.
QUESTION: What is human robot interaction?
In the context of HRI, Bensch et al. propose a model that treats interaction as "an interplay between human(s), robot(s), and environment." Goodrich and Schultz describe interaction as "the process of working together to accomplish a goal."
QUESTION: What does the paper say about communication?
The paper discusses a model developed by Weaver that focuses on communication systems such as telegraph, radio, and telephone systems. This model falls within the cybernetic tradition in communication theory. Transmission models, like the one described, are criticized for being linear and one-way. They exhibit epistemological biases by treating information like a physi